# Newton's Laws and Force Analysis

This notebook covers Newton's three laws of motion and force analysis using SymPy's mechanics package.

## Topics Covered:
- Newton's three laws of motion
- Free body diagrams
- Force equilibrium
- Friction and tension forces
- Connected systems

In [ ]:
import sympy as sp
from sympy.physics.mechanics import *
import sympy.physics.mechanics as me
import numpy as np
import matplotlib.pyplot as plt

# Initialize symbols
sp.init_printing()

# Define reference frame
N = ReferenceFrame('N')  # Inertial frame

# Define symbols
m, g, mu_s, mu_k = sp.symbols('m g mu_s mu_k', positive=True)
F, T, theta = sp.symbols('F T theta', real=True)
t = sp.symbols('t', real=True)

## Newton's Second Law: F = ma

In [ ]:
# Define a particle
particle = Particle('particle', mass=m)

# Position vector
x = sp.Function('x')(t)
y = sp.Function('y')(t)
r = x*N.x + y*N.y

# Velocity and acceleration
v = r.diff(t)
a = v.diff(t)

print("Particle kinematics:")
print(f"Position: {r}")
print(f"Velocity: {v}")
print(f"Acceleration: {a}")

# Apply Newton's second law
F_net = m * a
print(f"\nNet force (Newton's 2nd law): {F_net}")

## Example: Block on Inclined Plane

In [ ]:
# Define angle of incline
alpha = sp.symbols('alpha', real=True)

# Forces on the block
# Weight components
W_parallel = m * g * sp.sin(alpha)  # Down the incline
W_perpendicular = m * g * sp.cos(alpha)  # Into the incline

# Normal force
N_force = W_perpendicular  # Equilibrium in perpendicular direction

# Friction force (static case)
f_s_max = mu_s * N_force

# Friction force (kinetic case)
f_k = mu_k * N_force

print("Forces on inclined plane:")
print(f"Weight parallel to plane: {W_parallel}")
print(f"Weight perpendicular to plane: {W_perpendicular}")
print(f"Normal force: {N_force}")
print(f"Maximum static friction: {f_s_max}")
print(f"Kinetic friction: {f_k}")

## Equilibrium Analysis

In [ ]:
# Check if block remains stationary (static equilibrium)
# Condition: W_parallel <= f_s_max
equilibrium_condition = sp.simplify(W_parallel - f_s_max)
print(f"Equilibrium condition (W_parallel - f_s_max): {equilibrium_condition}")

# Find critical angle where block starts to slide
critical_angle = sp.solve(sp.Eq(W_parallel, f_s_max), alpha)[0]
print(f"\nCritical angle: {critical_angle}")
print(f"This means tan(α_critical) = μ_s")

# Numerical example
values = {m: 5, g: 9.81, mu_s: 0.4}
critical_angle_num = float(critical_angle.subs(values))
print(f"\nNumerical example (μ_s = 0.4):")
print(f"Critical angle: {np.degrees(critical_angle_num):.2f} degrees")

## Connected Systems: Atwood Machine

In [ ]:
# Define masses for Atwood machine
m1, m2 = sp.symbols('m1 m2', positive=True)
a = sp.symbols('a', real=True)  # Acceleration
T = sp.symbols('T', real=True)  # Tension in rope

# Forces on mass m1 (heavier, going down)
W1 = m1 * g
F_net1 = W1 - T  # Downward positive

# Forces on mass m2 (lighter, going up)
W2 = m2 * g
F_net2 = T - W2  # Upward positive

# Apply Newton's second law to both masses
eq1 = sp.Eq(F_net1, m1 * a)
eq2 = sp.Eq(F_net2, m2 * a)

print("Atwood machine equations:")
print(f"Mass 1: {eq1}")
print(f"Mass 2: {eq2}")

# Solve for acceleration and tension
solution = sp.solve([eq1, eq2], (a, T))
print(f"\nSolution:")
print(f"Acceleration: {solution[a]}")
print(f"Tension: {solution[T]}")

## Numerical Example of Atwood Machine

In [ ]:
# Numerical values
values_atwood = {m1: 3, m2: 1, g: 9.81}

a_num = float(solution[a].subs(values_atwood))
T_num = float(solution[T].subs(values_atwood))

print(f"Atwood machine numerical example (m1=3kg, m2=1kg):")
print(f"Acceleration: {a_num:.2f} m/s²")
print(f"Tension: {T_num:.2f} N")

# Check if the solution makes sense
print(f"\nVerification:")
print(f"m1*g - T = {3*9.81 - T_num:.2f} N")
print(f"m1*a = {3*a_num:.2f} N")
print(f"T - m2*g = {T_num - 1*9.81:.2f} N")
print(f"m2*a = {1*a_num:.2f} N")

## Spring-Mass System

In [ ]:
# Define spring constant
k = sp.symbols('k', positive=True)

# Position from equilibrium
x = sp.Function('x')(t)

# Spring force (Hooke's law)
F_spring = -k * x  # Negative because it opposes displacement

# Apply Newton's second law
m_spring = sp.symbols('m_spring', positive=True)
equation = sp.Eq(m_spring * sp.diff(x, t, 2), F_spring)

print("Spring-mass system:")
print(f"Spring force: {F_spring}")
print(f"Equation of motion: {equation}")

# Natural frequency
omega = sp.sqrt(k/m_spring)
print(f"\nNatural frequency: ω = {omega}")
print(f"Period: T = {2*sp.pi/omega}")

## Damped Oscillator

In [ ]:
# Define damping coefficient
b = sp.symbols('b', positive=True)

# Velocity
v = sp.diff(x, t)

# Damping force (opposes velocity)
F_damping = -b * v

# Total force on damped oscillator
F_total = F_spring + F_damping

# Equation of motion
damped_eq = sp.Eq(m_spring * sp.diff(x, t, 2), F_total)

print("Damped oscillator:")
print(f"Damping force: {F_damping}")
print(f"Total force: {F_total}")
print(f"Equation of motion: {damped_eq}")

# Standard form
standard_form = sp.simplify(damped_eq.lhs / m_spring - damped_eq.rhs / m_spring)
print(f"\nStandard form: {standard_form} = 0")

# Damping ratio
gamma = b / (2 * m_spring)
print(f"\nDamping coefficient: γ = {gamma}")
print(f"Natural frequency: ω₀ = {omega}")
print(f"Damping ratio: ζ = {gamma/omega}")

## Free Body Diagram Analysis

Let's create a systematic approach to free body diagrams:

In [ ]:
def analyze_forces(masses, forces, constraints=None):
    """
    Analyze forces on a system of particles
    masses: dictionary of particle names to masses
    forces: dictionary of particle names to force expressions
    constraints: list of constraint equations
    """
    results = {}
    
    for name, mass in masses.items():
        if name in forces:
            F_net = forces[name]
            # Newton's second law
            a = F_net / mass
            results[name] = {'mass': mass, 'force': F_net, 'acceleration': a}
    
    return results

# Example: Two connected blocks on horizontal surface
m_A, m_B = sp.symbols('m_A m_B', positive=True)
F_applied, f_friction = sp.symbols('F_applied f_friction', real=True)

# Forces on block A (being pulled)
F_A = F_applied - f_friction

# Forces on block B (connected to A)
F_B = f_friction  # Equal and opposite to friction on A

masses = {'A': m_A, 'B': m_B}
forces = {'A': F_A, 'B': F_B}

analysis = analyze_forces(masses, forces)

print("Connected blocks analysis:")
for name, result in analysis.items():
    print(f"\nBlock {name}:")
    print(f"  Mass: {result['mass']}")
    print(f"  Net force: {result['force']}")
    print(f"  Acceleration: {result['acceleration']}")

## Summary

This notebook covered:
- Newton's three laws of motion
- Force analysis on inclined planes
- Connected systems (Atwood machine)
- Spring-mass systems and damping
- Systematic free body diagram analysis

Key concepts:
- F = ma is the fundamental equation of dynamics
- Free body diagrams help identify all forces
- Constraint forces (tension, normal force) maintain system geometry
- Friction opposes relative motion